### ETL in Databricks

The notebook will be used as the source a daily job to refresh the pipeline (The whole notebook will be executed) and a dashboard will be created using the gold tables as source data.


In [0]:
%sql
-- Setup catalog if not exists
--CREATE CATALOG IF NOT EXISTS ws_banking_etl2;

-- Setup schemas for medallion architecture
CREATE SCHEMA IF NOT EXISTS ws_banking_etl2.bronze;
CREATE SCHEMA IF NOT EXISTS ws_banking_etl2.silver;
CREATE SCHEMA IF NOT EXISTS ws_banking_etl2.gold;

### Ingest from external location


#### Users_data.csv

In [0]:
users_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("abfss://blobcontainer@eltblobstorage.dfs.core.windows.net/users_data.csv")
)

#display(users_df.limit(10))
users_df.show(5)
print(f"users row count: {users_df.count()}")

+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
|  id|current_age|retirement_age|birth_year|birth_month|gender|             address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+----+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
| 825|         53|            66|      1966|         11|Female|       462 Rose Lane|   34.15|  -117.76|           $29278|       $59696|   $127613|         787|               5|
|1746|         53|            68|      1966|         12|Female|3606 Federal Boul...|   40.76|   -73.74|           $37891|       $77254|   $191349|         701|               5|
|1718|         81|            67|      1938|         11|Female|     766 Third Drive|   34.02|  -117.89|           $

In [0]:
users_df.write.format("delta").mode("overwrite").saveAsTable("bronze.users_data")

#### mcc_codes.json

In [0]:
# mcc.json is a multi-line single JSON object -> so spark cannot parse it -> throw _corrupt_record
mcc_raw = (
    spark.read
    .option("multiline", "true")
    .json("abfss://blobcontainer@eltblobstorage.dfs.core.windows.net/raw/mcc_codes.json")
)

# display(mcc_raw)
mcc_raw.show(5)
print(f"mcc_codes row count: {mcc_raw.count()}")

+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------+--------+------------------+--------------------+--------------------+--------------------+--------------------+------------------+--------------+--------------------+----------------+--------------------+--------------------+------------------+--------------------+---------+--------------------+------------+--------+---------------+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+-----

In [0]:
from pyspark.sql.functions import explode, create_map, lit, col

mcc_df = mcc_raw.select(
    explode(
        create_map(*[
            x for col_name in mcc_raw.columns
            for x in (lit(col_name), col(f"`{col_name}`"))
        ])
    ).alias("mcc_code", "description")
)

mcc_df.show(5)
print(f"mcc_codes row count: {mcc_df.count()}")

+--------+--------------------+
|mcc_code|         description|
+--------+--------------------+
|    1711|Heating, Plumbing...|
|    3000|          Steelworks|
|    3001|Steel Products Ma...|
|    3005|Miscellaneous Met...|
|    3006|Miscellaneous Fab...|
+--------+--------------------+
only showing top 5 rows
mcc_codes row count: 109


In [0]:
mcc_df.write.format("delta").mode("overwrite").saveAsTable("bronze.mcc_codes")

### Read JSON -  train_fraud_labels
One giant JSON object (a dictionary)

Top-level field: "target"

"target" contains a very large map of ID → "No"

Example:

{
  "target": {
    "10649266": "No",
    "23410063": "No",
    ...
  }
}

In [0]:
from pyspark.sql.functions import from_json, explode, col
from pyspark.sql.types import StructType, MapType, StringType, StructField

# Step 1: read JSON as raw text (safe)
fraud_text = spark.read.text("abfss://blobcontainer@eltblobstorage.dfs.core.windows.net/raw/train_fraud_labels.json")

# Step 2: define schema for top-level object
schema = StructType([
    StructField("target", MapType(StringType(), StringType()))
])

# Step 3: parse JSON per row("json_data" is a Struct containing "target"..."target" is a Map with key = ID, value = "No"/"Yes")
fraud_parsed = fraud_text.select(from_json(col("value"), schema).alias("json_data"))

# Step 4: explode the "target" map into rows
fraud_df = fraud_parsed.select(explode("json_data.target").alias("id", "target"))

# display(fraud_df.limit(10))
fraud_df.show(5)
print(f"fraud_labels row count: {fraud_df.count()}")

+--------+------+
|      id|target|
+--------+------+
|10649266|    No|
|23410063|    No|
| 9316588|    No|
|12478022|    No|
| 9558530|    No|
+--------+------+
only showing top 5 rows
fraud_labels row count: 8914963


In [0]:
fraud_df.write.format("delta").mode("overwrite").saveAsTable("bronze.train_fraud_labels")

##Ingest Using JDBC

In [0]:
# transactions_data
url = (
    "jdbc:sqlserver://ticdatabricks2.database.windows.net:1433;"
    "database=ticdatabase;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

df_transactions = (
    spark.read
    .format("jdbc")
    .option("url", url)
    .option("dbtable", "dbo.transactions_data")
    .option("user", "sqladmin")
    .option("password", "Password1")
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
    .load()
)

# display(df_transactions)
df_transactions.show(5)
print(f"transactions_data row count: {df_transactions.count()}")

+--------+-------------------+---------+-------+--------+-----------------+-----------+-------------+--------------+-------+----+--------------------+
|      id|               date|client_id|card_id|  amount|         use_chip|merchant_id|merchant_city|merchant_state|    zip| mcc|              errors|
+--------+-------------------+---------+-------+--------+-----------------+-----------+-------------+--------------+-------+----+--------------------+
|12887527|2013-06-14 15:20:00|      582|   5233| 11.1800|Swipe Transaction|      24504|   Fort Wayne|            IN|46803.0|4214|                NULL|
|12887529|2013-06-14 15:20:00|     1020|   1224|-58.0000|Swipe Transaction|      61195|     Evanston|            IL|60201.0|5541|                NULL|
|12887530|2013-06-14 15:20:00|     1499|   3991|-53.0000|Swipe Transaction|      43293|       Nassau|   The Bahamas|   NULL|5499|                NULL|
|12887532|2013-06-14 15:21:00|      811|   2079| 84.7600|Swipe Transaction|      52466|Overlan

In [0]:
df_transactions.write.format("delta").mode("overwrite").saveAsTable("bronze.transactions_data")

In [0]:
# cards_data
url = (
    "jdbc:sqlserver://ticdatabricks2.database.windows.net:1433;"
    "database=ticdatabase;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

df_cards = (spark.read
    .format("jdbc")
    .option("url", url)
    .option("dbtable", "dbo.cards_data")
    .option("user", "sqladmin")
    .option("password", "Password1")
    .load()
)

# display(df_cards)
df_cards.show(5)
print(f"cards_data row count: {df_cards.count()}")


+----+---------+----------+---------------+----------------+-------+---+--------+----------------+------------+--------------+---------------------+----------------+
|  id|client_id|card_brand|      card_type|     card_number|expires|cvv|has_chip|num_cards_issued|credit_limit|acct_open_date|year_pin_last_changed|card_on_dark_web|
+----+---------+----------+---------------+----------------+-------+---+--------+----------------+------------+--------------+---------------------+----------------+
|4524|      825|      Visa|          Debit|4344676511950444|12/2022|623|     YES|               2|  24295.0000|       09/2002|                 2008|              No|
|2731|      825|      Visa|          Debit|4956965974959986|12/2020|393|     YES|               2|  21968.0000|       04/2014|                 2014|              No|
|3701|      825|      Visa|          Debit|4582313478255491|02/2024|719|     YES|               2|  46414.0000|       07/2003|                 2004|              No|
|  4

In [0]:
df_cards.write.format("delta").mode("overwrite").saveAsTable("bronze.cards_data")


###Verify bronze tables

In [0]:
spark.sql("SHOW TABLES IN bronze").show()


+--------+------------------+-----------+
|database|         tableName|isTemporary|
+--------+------------------+-----------+
|  bronze|        cards_data|      false|
|  bronze|         mcc_codes|      false|
|  bronze|train_fraud_labels|      false|
|  bronze| transactions_data|      false|
|  bronze|        users_data|      false|
+--------+------------------+-----------+



In [0]:
spark.read.table("bronze.transactions_data").printSchema()
spark.read.table("bronze.cards_data").printSchema()
spark.read.table("bronze.users_data").printSchema()
spark.read.table("bronze.mcc_codes").printSchema()
spark.read.table("bronze.train_fraud_labels").printSchema()


root
 |-- id: integer (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_id: short (nullable = true)
 |-- amount: decimal(19,4) (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: double (nullable = true)
 |-- mcc: short (nullable = true)
 |-- errors: string (nullable = true)

root
 |-- id: short (nullable = true)
 |-- client_id: short (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: long (nullable = true)
 |-- expires: string (nullable = true)
 |-- cvv: short (nullable = true)
 |-- has_chip: string (nullable = true)
 |-- num_cards_issued: short (nullable = true)
 |-- credit_limit: decimal(19,4) (nullable = true)
 |-- acct_open_date: string (nullable = true)
 |-- year_pin_last_changed: short (nullable = true)
 |--